# Using the BaseComposite Module in baseobjects.composition

## Introduction

The Composite design pattern allows you to compose objects into tree structures to represent part-whole hierarchies. `BaseComposite` and `BaseComponent` provide the foundational building blocks for implementing this pattern in your projects.

`BaseComposite` is a basic composite object that can contain multiple component objects.
`BaseComponent` is a basic component object designed to be used within a `BaseComposite`.

This tutorial covers:
- Creating a basic component class
- Defining a composite class with default components
- Adding and removing components dynamically
- Bidirectional communication between composites and components
- Serialization (Pickling) of composite structures

**Prerequisites:**
- Basic familiarity with Python
- Installed package: `baseobjects`

### Table of Contents

- [Importing the Module](#Importing-the-Module)
- [Core Functionality](#Core-Functionality)
- [Module Interaction](#Module-Interaction)
- [Advanced Features](#Advanced-Features)
- [Examples](#Examples)
- [API Highlights](#API-Highlights)
- [Troubleshooting / FAQs](#Troubleshooting-/-FAQs)
- [Conclusion and Next Steps](#Conclusion-and-Next-Steps)

## Importing the Module

First, we need to import `BaseComposite` and `BaseComponent` from `baseobjects.composition`.


In [ ]:
from typing import Any, ClassVar
from baseobjects.composition import BaseComposite, BaseComponent


## Core Functionality

To create a component, you inherit from `BaseComponent`. Components can interact with their parent composite through the `self.composite` property.

### Key Concepts

1. **BaseComponent**: The base class for all components. It maintains a reference to its parent composite.
2. **BaseComposite**: The base class for composite objects. It manages a collection of components.
3. **Bidirectional Communication**: Components can access the composite's state, and the composite can invoke component methods.


In [ ]:
class MathComponent(BaseComponent):
    """A component that provides mathematical operations for a composite."""

    def add(self, a: int, b: int) -> int:
        """Adds two numbers and updates the composite's result."""
        result = a + b
        self.composite.result = result
        return result

class LoggingComponent(BaseComponent):
    """A component that logs operations performed on a composite."""

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.log = []

    def log_operation(self, operation: str, *args: Any) -> None:
        """Logs an operation performed on the composite."""
        log_entry = f"{operation}({', '.join(str(arg) for arg in args)}) = {self.composite.result}"
        self.log.append(log_entry)
        print(f"Logged: {log_entry}")


### Defining the Composite

A composite class inherits from `BaseComposite`. You can define `default_component_types` to specify which components should be automatically created upon instantiation.


In [ ]:
class Calculator(BaseComposite):
    """A composite calculator that uses components for different operations."""

    # Define default components
    default_component_types: ClassVar[dict[str, tuple[type[BaseComponent], dict[str, Any]]]] = {
        "math": (MathComponent, {}),
        "logging": (LoggingComponent, {}),
    }

    def __init__(self, *args, **kwargs):
        self.result = 0
        super().__init__(*args, **kwargs)


### Basic Usage

When you instantiate the `Calculator`, it automatically creates instances of `MathComponent` and `LoggingComponent`.


In [ ]:
# Create a calculator composite
calculator = Calculator()

# Access components via the 'components' dictionary
math = calculator.components["math"]
logger = calculator.components["logging"]

# Use the math component
math.add(10, 5)
print(f"Calculator result: {calculator.result}")

# Use the logging component
logger.log_operation("add", 10, 5)


## Module Interaction

`BaseComposite` and `BaseComponent` are designed to work together. The composite manages the lifecycle of components, and components have a back-reference to the composite.


In [ ]:
# Demonstrate bidirectional link
print(f"Math component's composite: {math.composite is calculator}")
print(f"Logger component's composite: {logger.composite is calculator}")


## Advanced Features

### Dynamic Component Management

You can add or remove components at runtime using `add_component`, `create_component`, and `remove_component`.


In [ ]:
class FormattingComponent(BaseComponent):
    """A component that formats the result."""
    def format_result(self) -> str:
        return f"Result is: {self.composite.result}"

# Create and add a component manually
formatter = FormattingComponent()
calculator.add_component("formatting", formatter)

# Or use create_component to let the composite instantiate it
# calculator.create_component("formatting", FormattingComponent)

print(calculator.components["formatting"].format_result())

# Remove a component
removed = calculator.remove_component("formatting")
print(f"Removed component: {type(removed).__name__}")
print(f"Remaining components: {list(calculator.components.keys())}")


### Serialization (Pickling)

`BaseComposite` and `BaseComponent` are designed to be picklable. `BaseComponent` uses a weak reference to its composite to avoid circular references, but handles the reconstruction of these references during deserialization.


In [ ]:
import pickle

# Set a state in the composite
calculator.result = 42

# Serialize
data = pickle.dumps(calculator)

# Deserialize
new_calculator = pickle.loads(data)

print(f"Deserialized result: {new_calculator.result}")
print(f"Components: {list(new_calculator.components.keys())}")

# The link between component and composite is restored
new_math = new_calculator.components["math"]
print(f"Component's composite is restored: {new_math.composite is new_calculator}")


## Examples

Here is a complete example of a simple data processing pipeline using components.


In [ ]:
class DataCleaner(BaseComponent):
    def clean(self, data: str) -> str:
        return data.strip()

class DataProcessor(BaseComposite):
    default_component_types = {"cleaner": (DataCleaner, {})}
    def process(self, data: str) -> str:
        return self.components["cleaner"].clean(data)

processor = DataProcessor()
print(f"Processed data: '{processor.process('  some data  ')}'")


## API Highlights

- **`BaseComposite`**: The main class for managing components.
  - `components`: A dictionary of attached components.
  - `add_component(name, component)`: Attaches an existing component.
  - `create_component(name, component_type, **kwargs)`: Instantiates and attaches a component.
  - `remove_component(name)`: Detaches a component.
- **`BaseComponent`**: The base class for components.
  - `composite`: A property that returns the parent composite.

## Troubleshooting / FAQs

- **Problem**: `AttributeError: 'NoneType' object has no attribute 'result'` inside a component.
  - **Solution**: This usually means the component has not been attached to a composite yet. Ensure `add_component` or `create_component` has been called, or that it was included in `default_component_types`.

- **Problem**: Circular reference issues when pickling.
  - **Solution**: `BaseComponent` uses `weakref` for its `composite` link. If you implement custom `__getstate__` or `__setstate__`, ensure you handle the composite reference correctly or rely on the base implementation.

## Conclusion and Next Steps

`BaseComposite` and `BaseComponent` provide a flexible way to build complex objects from smaller, reusable parts while maintaining clean separation of concerns and easy management of object lifecycles.

- **Next**: Learn about `BaseDispatchingComposite` for dynamic component selection.
- **Reference**: See `src/baseobjects/composition/basecomposite.py` for implementation details.
